<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-07/NB07_embodiment.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 07: Embodiment, Situatedness and the Extended Mind

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-07/lab.html>.

## Overview

Classical AI treated intelligence as reasoning over internal representations. This week examines the alternative that intelligence arises from bodies acting in environments: behaviour-based robotics, Braitenberg's synthetic psychology, enactivism and the thesis that the mind extends beyond the skull [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to explain Brooks's case against central representation and the subsumption architecture, to derive the behaviour of Braitenberg vehicles from their wiring, to describe the enactive and morphological views of cognition, and to evaluate the parity argument for the extended mind.

## Intelligence without representation

Brooks argued that AI had focused on the wrong problems [1]. Instead of building disembodied reasoners with central world models, researchers should build complete creatures that act in the real world, starting with simple behaviours and adding layers. In his subsumption architecture, each layer connects perception directly to action, such as avoiding obstacles or wandering, and higher layers can suppress or override lower ones. The robots used the world itself as their source of information instead of maintaining detailed internal models.

Moravec observed a related asymmetry, often called Moravec's paradox [4]. Tasks that people find intellectually hard, such as chess, turned out to be comparatively easy for computers, while tasks that people find effortless, such as perceiving and moving, turned out to be very hard. Evolution spent far longer refining sensorimotor skills than abstract reasoning.

## Synthetic psychology

Braitenberg described a series of imaginary vehicles, each with sensors connected to motors by simple wiring [2]. A vehicle with two light sensors, each driving the wheel on its own side more strongly when light is brighter, turns away from light and seems to fear it. Crossing the wires makes it turn towards the light and rush at it, as if aggressive. Making the connections inhibitory produces a vehicle that approaches the light and slows down, facing it as if in love, or one that approaches and then turns away, like an explorer. Figure 7.1 shows these trajectories.

Braitenberg drew a methodological moral, the law of uphill analysis and downhill invention: It is easier to build a mechanism that produces a behaviour than to infer the mechanism from the behaviour. Observers readily attribute emotions and intentions to the vehicles, even though the mechanism is transparent to the designer. The lab lets students rewire the vehicles and change their morphology, and Week 14 returns to the attribution of intentions.

![Figure 7.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-07/figures/w07_fig1.png)

*Figure 7.1. Trajectories of four Braitenberg vehicles that differ only in whether their sensor-motor connections are crossed and whether they excite or inhibit [2].*

## The embodied and enactive mind

Varela, Thompson and Rosch proposed that cognition is enaction: the bringing forth of a world through the history of structural coupling between an organism and its environment, grounded in sensorimotor capacities [5]. Cognition, on this view, is not the representation of a pre-given world by a pre-given mind. Pfeifer and Bongard developed the engineering side of the idea and argued that the shape and material of a body take over part of the work that would otherwise require computation, a phenomenon they called morphological computation [6]. A change in the angle of a vehicle's sensors, as in the lab, can change its behaviour as much as a change in its wiring.

## The extended mind

Clark and Chalmers asked where the mind stops and the rest of the world begins [3]. Their example contrasts Inga, who remembers the address of a museum, with Otto, who has a memory impairment and consults a notebook he always carries. They proposed a parity principle: If a part of the world functions as a process that we would call cognitive if it occurred in the head, then that part of the world is part of the cognitive process. Otto's notebook, on this view, is part of his memory.

The thesis raises questions about AI systems that rely on external tools and stores, and about people who rely on AI systems. Critics object that the notebook lacks the integration and automatic trust that mark biological memory. The notebook of this week simulates agents with limited internal memory that can consult an external store, which makes the parity principle a question about performance and cost.

> **Pause and reflect.** Is a search engine you consult daily part of your memory in the sense of Clark and Chalmers? Which of their conditions does it meet?

# Hands-on lab

The notebook simulates Braitenberg vehicles, builds a small subsumption controller whose layers can be switched on and off, and models an agent with limited internal memory that can consult an external store, in the spirit of the parity principle [1, 2, 3].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Braitenberg vehicles

In [ ]:
def vehicle(kind, light=(4.0, 3.0), start=(-6.0, -4.0), heading=0.3, phi=0.5, steps=700, dt=0.05):
    x, y, th = start[0], start[1], heading
    path = []
    for _ in range(steps):
        path.append((x, y))
        il, ir = [1.0 / (1.0 + ((x + 0.5 * np.cos(th + s * phi) - light[0]) ** 2 + (y + 0.5 * np.sin(th + s * phi) - light[1]) ** 2) / 4.0) for s in (1, -1)]
        k, v0 = 6.0, 2.0
        vl, vr = {"2a": (k * il, k * ir), "2b": (k * ir, k * il), "3a": (v0 - k * il, v0 - k * ir), "3b": (v0 - k * ir, v0 - k * il)}[kind]
        vl, vr = max(vl, 0.0), max(vr, 0.0)
        th += (vr - vl) * dt
        x += (vl + vr) / 2 * np.cos(th) * dt
        y += (vl + vr) / 2 * np.sin(th) * dt
    return np.array(path)

for kind in ["2a", "2b", "3a", "3b"]:
    p = vehicle(kind)
    plt.plot(p[:, 0], p[:, 1], label=kind)
plt.plot(4, 3, "*", ms=15, color="orange"); plt.legend(); plt.gca().set_aspect("equal"); plt.title("Four wirings, one light"); plt.show()

### Your turn, exercise 1

Compute the final distance to the light for vehicle 3a with the default settings. Then compute it for vehicle 3b and describe in two sentences how the two inhibitory wirings differ in behaviour.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _final_distance_reference(kind="3a", phi=0.5):
    p = vehicle(kind, phi=phi)
    return float(np.hypot(p[-1, 0] - 4.0, p[-1, 1] - 3.0))

In [ ]:
final_distance = None  # your computation for vehicle 3a with the defaults
check("Exercise 1", final_distance, _final_distance_reference())

## 2. A subsumption controller

A point robot moves in a square arena towards a light. Layer 0 wanders, layer 1 avoids walls and suppresses wandering near them, and layer 2 steers towards the light when it is close enough to sense it.

In [ ]:
def run_robot(layers, steps=600, seed=0):
    rng = np.random.default_rng(seed)
    x, y, th = -8.0, -8.0, 0.0
    light, collisions = np.array([7.0, 7.0]), 0
    for _ in range(steps):
        turn = rng.normal(0, 0.4) if 0 in layers else 0.0
        if 2 in layers and np.hypot(light[0] - x, light[1] - y) < 9:
            want = np.arctan2(light[1] - y, light[0] - x)
            turn = 0.3 * np.angle(np.exp(1j * (want - th)))
        near_wall = max(abs(x), abs(y)) > 8.5
        if 1 in layers and near_wall:
            turn = np.angle(np.exp(1j * (np.arctan2(-y, -x) - th)))
        th += turn
        x, y = x + 0.2 * np.cos(th), y + 0.2 * np.sin(th)
        if max(abs(x), abs(y)) > 10:
            collisions += 1
            x, y = np.clip(x, -10, 10), np.clip(y, -10, 10)
        if np.hypot(light[0] - x, light[1] - y) < 0.5:
            return True, collisions
    return False, collisions

for layers in [(0,), (0, 1), (0, 1, 2)]:
    results = [run_robot(layers, seed=s) for s in range(30)]
    print(f"layers {layers}: reached the light in {sum(r[0] for r in results)} of 30 runs, mean collisions {np.mean([r[1] for r in results]):.1f}")

### Your turn, exercise 2

Compute the mean number of wall collisions over seeds 0 to 29 for the controller with layers (0, 2), that is, without obstacle avoidance. Explain in two sentences what the comparison shows about layered control.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _collisions_reference():
    return float(np.mean([run_robot((0, 2), seed=s)[1] for s in range(30)]))

In [ ]:
mean_collisions = None  # your computation
check("Exercise 2", mean_collisions, _collisions_reference())

## 3. Memory inside and outside the head

An agent must recall items from a list. Its internal memory keeps only the last four items. An external store keeps everything but costs one time unit per lookup.

In [ ]:
def recall_accuracy(n_items, external, trials=2000):
    hits, cost = 0, 0
    for _ in range(trials):
        items = rng_mem.integers(0, 1000, n_items)
        probe = rng_mem.integers(n_items)
        internal = set(items[-4:])
        if items[probe] in internal:
            hits += 1
        elif external:
            hits += 1; cost += 1
    return hits / trials, cost / trials

rng_mem = np.random.default_rng(SEED)
for n in [4, 8, 16]:
    print(n, "items: internal only", recall_accuracy(n, False), " with external store", recall_accuracy(n, True))

Under the parity principle, the external store counts as part of the memory system if it plays the same role as internal memory [3]. The simulation shows the functional trade-off; whether the store is part of the mind is the philosophical question.

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-07/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which Braitenberg vehicle surprised you most, and which descriptions came to mind before you checked the wiring?
2. Could a language model without a body be intelligent in the sense of Brooks or the enactivists? Give their likely answers and your own.
3. Identify a tool in your research practice that might count as part of your extended mind. Which conditions of Clark and Chalmers does it meet?

## Weekly task and submission

Write about 500 words that apply the embodied and extended views of cognition to an AI system of your choice. Use results from the lab or the notebook as examples and state whether the system's intelligence is located in the model, the body, the environment or their coupling. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Embodiment in the age of foundation models.** Evaluate whether robots controlled by large pretrained models vindicate or challenge the embodied and enactive views of cognition [1, 5, 6].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## Midterm capstone

This week closes the first half of the course with the [Midterm Capstone: Machines, Tests and Meaning](https://github.com/utkukose/philosophy-of-ai-NB-lecture/blob/main/exams/midterm/README.md).

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the midterm capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 7, with the report and all code files attached or linked.

## References

[1] Brooks, R. A. (1991). Intelligence without representation. *Artificial Intelligence*, 47(1-3), 139-159. <https://doi.org/10.1016/0004-3702(91)90053-M>

[2] Braitenberg, V. (1984). *Vehicles: Experiments in Synthetic Psychology*. MIT Press.

[3] Clark, A., & Chalmers, D. (1998). The extended mind. *Analysis*, 58(1), 7-19. <https://doi.org/10.1093/analys/58.1.7>

[4] Moravec, H. (1988). *Mind Children: The Future of Robot and Human Intelligence*. Harvard University Press.

[5] Varela, F. J., Thompson, E., & Rosch, E. (1991). *The Embodied Mind: Cognitive Science and Human Experience*. MIT Press.

[6] Pfeifer, R., & Bongard, J. (2006). *How the Body Shapes the Way We Think: A New View of Intelligence*. MIT Press.